# Demo: Secure Tabular Synthetic Data Generation Platform
### Nền tảng Sinh Dữ liệu Dạng Bảng Tổng hợp Bảo mật (CTVAE, CTGAN & Tabular Diffusion)

Notebook này hướng dẫn bạn chạy từng bước của quy trình **End-to-End Pipeline** thực tế trên giấy và trong mã nguồn của đề tài:
1. **Stage 1 & 2**: Đọc dữ liệu thô, loại bỏ cột nhạy cảm (PII), điền khuyết (Imputation) kết hợp chỉ báo khuyết thiếu, mã hóa đặc trưng và chuẩn hóa.
2. **Stage 3 & 4**: Nạp checkpoint mô hình sinh sâu đã huấn luyện (CTVAE, CTGAN, hoặc Diffusion), chạy quá trình lấy mẫu ổn định (Sampling với `clip_denoised`), giải mã ngược và áp dụng luật ràng buộc logic nghiệp vụ (`ConstraintsEngine`).
3. **Audit Framework**: Chạy kiểm toán chất lượng đa chiều đo lường Độ trung thực (Fidelity), Bảo mật (Privacy - MIA, DCR) và Độ hữu dụng học máy (Utility - TSTR vs TRTR) kèm trực quan hóa kết quả.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Thiết lập thư mục làm việc chính của dự án
project_root = os.path.abspath(os.path.join(os.getcwd()))
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Thư mục dự án: {project_root}")

## Bước 1: Khởi tạo và Tiền xử lý dữ liệu (Stage 1 & 2)
Chúng ta sẽ nạp cấu hình YAML của tập dữ liệu `telco_customer_churn`, thực hiện khử định danh cột `customerID`, tạo cột chỉ báo khuyết thiếu và chuẩn hóa dữ liệu thô.

In [ ]:
from src.config.config_loader import ConfigLoader
from src.preprocessing.pipeline import PreprocessingPipeline

dataset_name = "telco_customer_churn" # Bạn có thể đổi sang "adult_income"
data_path = os.path.join("data", "Telco-Customer-Churn.csv")

# 1. Load cấu hình cấu trúc dữ liệu
config = ConfigLoader.load_config(dataset_name)
print(f"[Config] Tên tập dữ liệu: {dataset_name}")
print(f"[Config] Mô hình mặc định: {config.model.model_type.upper()}")
print(f"[Config] Cột mục tiêu (Target): {config.ingestion.target_column}")

# 2. Khởi tạo Pipeline tiền xử lý
pipeline = PreprocessingPipeline(dataset_name)
df_raw = pipeline.load_data(data_path)
print(f"\n[Stage 1] Kích thước dữ liệu thô: {df_raw.shape}")

# 3. Chạy fit_transform (Stage 2)
df_preprocessed = pipeline.fit_transform(df_raw)
pipeline.save_artifacts() # Lưu tệp preprocessing_pipeline.joblib kèm mã SHA256
print(f"[Stage 2] Kích thước dữ liệu sau tiền xử lý: {df_preprocessed.shape}")

## Bước 2: Sinh dữ liệu tổng hợp từ Checkpoint (Stage 4)
Thay vì tốn thời gian huấn luyện lại (mất 15 phút), chúng ta sẽ nạp trực tiếp checkpoint mô hình đã huấn luyện hoàn chỉnh trong thư mục `artifacts/` để thực hiện lấy mẫu sinh dữ liệu siêu tốc (chỉ mất 3 giây).

In [ ]:
from src.inference.sampler import SyntheticSampler

# Bạn có thể lựa chọn mô hình để sinh thử: "diffusion", "ctvae", "ctgan"
model_type = "diffusion"
print(f"Khởi tạo bộ sinh dữ liệu bằng mô hình: {model_type.upper()}")

sampler = SyntheticSampler(
    model_type=model_type,
    dataset_name=dataset_name
)
sampler.load() # Nạp model weights .pt và pipeline.joblib đã lưu

# Sinh dữ liệu tổng hợp (áp dụng clip_denoised, giải mã ngược và kẹp biên tự động)
df_synthetic = sampler.generate(n_rows=1000)
print(f"\nSinh dữ liệu thành công! Kích thước dữ liệu tổng hợp: {df_synthetic.shape}")

# Hiển thị 5 dòng dữ liệu tổng hợp đầu tiên
display(df_synthetic.head())

## Bước 3: Kiểm toán chất lượng đa chiều (Audit Framework)
Chúng ta sẽ chạy bộ đánh giá chất lượng `EvaluationSuite` để so sánh trực tiếp dữ liệu giả vừa sinh ra với tập dữ liệu thật ban đầu.

In [ ]:
from src.evaluation.orchestrator import EvaluationSuite

print("Bắt đầu chạy kịch bản kiểm toán đa chiều...")
suite = EvaluationSuite(dataset_name=dataset_name)

target_col = config.ingestion.target_column
sensitive_col = config.ingestion.quasi_identifiers[0] if config.ingestion.quasi_identifiers else ""

# Chạy kiểm toán
results = suite.run_evaluation(
    real_df=df_raw,
    synth_df=df_synthetic,
    target_col=target_col,
    sensitive_col=sensitive_col
)

print("\n" + "="*65)
print(f"        KẾT QUẢ KIỂM TOÁN CHẤT LƯỢNG MÔ HÌNH {model_type.upper()}")
print("="*65)
print(f"1. FIDELITY (Độ trung thực thống kê):")
print(f"   - JSD biên trung bình (Avg JSD)      : {results['fidelity']['avg_js']:.4f} (càng gần 0 càng tốt)")
print(f"   - Lệch tương quan chéo (Corr Diff)    : {results['fidelity']['corr_diff']:.4f} (càng gần 0 càng tốt)")
print(f"\n2. PRIVACY (An toàn bảo mật):")
print(f"   - Tấn công thành viên (MIA AUC-ROC) : {results['privacy']['mia_auc']:.4f} (gần 0.50 là bảo mật tốt nhất)")
print(f"   - Tỷ lệ rò rỉ bản ghi trùng (DCR Leak): {results['privacy']['dcr_leakage_pct']:.2f}%")
print(f"   - Khoảng cách bản ghi gần nhất (DCR Mean): {results['privacy']['dcr_mean']:.4f}")
print(f"\n3. UTILITY (Độ hữu dụng cho Machine Learning - TSTR vs TRTR):")
rf_metrics = results['utility']['metrics']['RandomForest']
print(f"   - RandomForest F1-Macro TSTR (Giả)   : {rf_metrics['tstr_f1']:.4f}")
print(f"   - RandomForest F1-Macro TRTR (Thật)  : {rf_metrics['trtr_f1']:.4f}")
gb_metrics = results['utility']['metrics']['GradientBoosting']
print(f"   - GradBoosting F1-Macro TSTR (Giả)   : {gb_metrics['tstr_f1']:.4f}")
print(f"   - GradBoosting F1-Macro TRTR (Thật)  : {gb_metrics['trtr_f1']:.4f}")
print("="*65)

## Bước 4: Trực quan hóa so sánh phân phối & Tương quan chéo
Hiển thị các biểu đồ overlay phân phối biên và ma trận tương quan được sinh tự động bởi hệ thống.

In [ ]:
plots_dir = suite.plots_dir
print(f"Thư mục chứa biểu đồ kết xuất: {plots_dir}\n")

# 1. Hiển thị biểu đồ so sánh tương quan chéo
corr_plot = os.path.join(plots_dir, "correlation_comparison.png")
if os.path.exists(corr_plot):
    plt.figure(figsize=(15, 8))
    img = plt.imread(corr_plot)
    plt.imshow(img)
    plt.axis('off')
    plt.title("So sánh Ma Trận Tương Quan Chéo (Real vs Synthetic)", fontsize=14, fontweight='bold')
    plt.show()

# 2. Hiển thị lưới biểu đồ phân phối biên
dist_plot = os.path.join(plots_dir, "distributions_grid.png")
if os.path.exists(dist_plot):
    plt.figure(figsize=(15, 12))
    img = plt.imread(dist_plot)
    plt.imshow(img)
    plt.axis('off')
    plt.title("Biểu đồ Đè phủ Phân Phối Biên của từng cột (Real vs Synthetic)", fontsize=14, fontweight='bold')
    plt.show()